# Ergodic uncertainty controller

The greedy baseline visits one maximum at a time. An ergodic controller instead makes the long-run spatial statistics of the sensor trajectory match the complete uncertainty distribution. This implementation follows the controller in the original `pac_box_fun` prototype.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from box_gym import BoxGym, candidate_uncertainty

In [ ]:
class ErgodicController:
    def __init__(self, env, grid_size=50, num_modes=10, integration_dt=0.01):
        self.env = env
        self.integration_dt = integration_dt
        self.dx = 1.0 / (grid_size - 1)

        axis = np.linspace(0.0, 1.0, grid_size)
        self.grid_x, self.grid_y = np.meshgrid(axis, axis)
        self.grid = np.column_stack((self.grid_x.ravel(), self.grid_y.ravel()))
        kx, ky = np.meshgrid(np.arange(num_modes), np.arange(num_modes))
        self.modes = np.column_stack((kx.ravel(), ky.ravel()))
        self.weights = (1.0 + np.linalg.norm(self.modes, axis=1)) ** -1.5

        raw_basis = np.prod(
            np.cos(np.pi * self.grid[:, None, :] * self.modes[None, :, :]),
            axis=2,
        )
        self.normalizers = np.sqrt((raw_basis**2).sum(axis=0) * self.dx**2)
        self.grid_basis = raw_basis / self.normalizers
        self.trajectory_coefficients = np.zeros(len(self.modes))
        self.target_coefficients = np.zeros(len(self.modes))
        self.time = 0.0
        self.entropy = np.zeros((grid_size, grid_size))

    def basis_at(self, position):
        return (
            np.prod(np.cos(np.pi * self.modes * position), axis=1)
            / self.normalizers
        )

    def update_target(self, rectangles):
        _, _, self.entropy = candidate_uncertainty(
            rectangles, self.grid_x.shape[0]
        )
        density = self.entropy.ravel().copy()
        mass = density.sum() * self.dx**2
        if mass < 1e-12:
            density.fill(1.0)
            mass = density.sum() * self.dx**2
        density /= mass
        self.target_coefficients = (
            self.grid_basis * density[:, None]
        ).sum(axis=0) * self.dx**2

    def action(self, observation):
        self.update_target(observation["pred_boxes"])
        start = observation["sensor_pos"].astype(float)
        simulated = start.copy()
        coefficients = self.trajectory_coefficients.copy()
        simulated_time = self.time
        steps = round(self.env.dt / self.integration_dt)

        for _ in range(steps):
            basis = self.basis_at(simulated)
            coefficients += self.integration_dt * basis
            gradient = np.column_stack((
                -np.pi * self.modes[:, 0]
                * np.sin(np.pi * self.modes[:, 0] * simulated[0])
                * np.cos(np.pi * self.modes[:, 1] * simulated[1]),
                -np.pi * self.modes[:, 1]
                * np.cos(np.pi * self.modes[:, 0] * simulated[0])
                * np.sin(np.pi * self.modes[:, 1] * simulated[1]),
            )) / self.normalizers[:, None]
            mismatch = (
                coefficients / (simulated_time + self.integration_dt)
                - self.target_coefficients
            )
            direction = -(self.weights * mismatch) @ gradient

            # Blend toward the centre only near the domain boundary.
            boundary_weight = (
                np.tanh(20 * (simulated - 0.02)) / 2
                + np.tanh(20 * (0.98 - simulated)) / 2
            )
            centre_pull = (
                -np.tanh(20 * (simulated - 0.02)) / 2
                + np.tanh(20 * (0.98 - simulated)) / 2
            )
            direction /= np.linalg.norm(direction) + 1e-12
            centre_pull /= np.linalg.norm(centre_pull) + 1e-12
            velocity = self.env.max_velocity * (
                direction * boundary_weight + centre_pull * (1 - boundary_weight)
            )
            simulated = np.clip(
                simulated + self.integration_dt * velocity, 0.0, 1.0
            )
            simulated_time += self.integration_dt

        self.trajectory_coefficients += self.env.dt * self.basis_at(start)
        self.time += self.env.dt
        action = simulated - start
        action /= np.linalg.norm(action) + 1e-12
        return (action * self.env.max_velocity).astype(np.float32)

In [ ]:
env = BoxGym(
    sensor_box_size=0.12,
    num_sensor_samples=4,
    max_velocity=0.25,
    inference_num=100,
)
obs, info = env.reset(seed=12)
controller = ErgodicController(env)
trajectory = [obs["sensor_pos"].copy()]

for step in range(300):
    action = controller.action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    trajectory.append(obs["sensor_pos"].copy())
    if terminated or truncated:
        break

controller.update_target(obs["pred_boxes"])
uncertainty_map = (controller.grid_x, controller.grid_y, controller.entropy)
print(f"Ran {step + 1} steps; final uncertainty: {info['uncertainty']:.5f}")

fig, ax = plt.subplots(figsize=(7, 7))
env.plot(
    ax,
    uncertainty_map=uncertainty_map,
    trajectory=np.asarray(trajectory),
)
ax.set_title("Ergodic uncertainty coverage")
plt.show()
env.close()